# DOCdemo 納品URL レビュー

**生成日**: 2026-05-11  
**目的**: DOCdemoカジュアル面談AIの納品URLを一覧表示し、各社のデモを確認するためのノートブック。

## 使い方
1. このノートブックを **Colab で開く**(共有メールから「Open with Colab」)
2. ランタイム → すべて実行 (`Ctrl+F9`)
3. 表の納品URLをクリックして各社のチャット画面を確認

## データソース
同フォルダの `delivery_snapshot_20260511.csv` を読み込みます (Google Drive にマウント)。

## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. CSV 読み込み
Driveのフォルダパスは、共有された場所に合わせて変更してください。

In [ ]:
import pandas as pd

# Drive 上の CSV パス (共有先で変わる場合は調整)
CSV_PATH = '/content/drive/MyDrive/DOCdemo_Deliveries/delivery_snapshot_20260511.csv'

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f'読み込み件数: {len(df)}社')
df.head()

## 3. ステータス別サマリ

In [ ]:
status_counts = df['ステータス'].value_counts()
print('--- ステータス別件数 ---')
for s, c in status_counts.items():
    print(f'  {s}: {c}社')

## 4. 納品URL 一覧 (完了企業のみ・リンク化)

In [ ]:
from IPython.display import HTML, display

completed = df[df['ステータス'] == '完了'][['企業名', '企業ID', '納品URL']].copy()

def link_html(url):
    if pd.isna(url) or not url:
        return ''
    return f'<a href="{url}" target="_blank">{url}</a>'

completed['納品URL'] = completed['納品URL'].apply(link_html)

html = completed.to_html(escape=False, index=False)
display(HTML(
    '<style>table {font-size:14px; border-collapse:collapse;}'
    'th, td {padding:8px; border:1px solid #ccc;}'
    'a {color:#1a73e8; text-decoration:none;}</style>'
    + html
))

## 5. 全企業一覧 (詳細)

In [ ]:
# 主要列のみ抜粋
view = df[['企業名', 'ホームページURL', '企業ID', '納品URL', 'ステータス', 'エラー詳細']].copy()

def link_optional(url):
    if pd.isna(url) or not url:
        return ''
    return f'<a href="{url}" target="_blank">{url}</a>'

view['ホームページURL'] = view['ホームページURL'].apply(link_optional)
view['納品URL'] = view['納品URL'].apply(link_optional)

html = view.to_html(escape=False, index=False)
display(HTML(
    '<style>table {font-size:13px; border-collapse:collapse;}'
    'th, td {padding:6px 10px; border:1px solid #ccc; vertical-align:top;}'
    'a {color:#1a73e8;}</style>'
    + html
))

---
## 補足
- **企業ID** は各社の公式ドメインから自動抽出した識別子 (例: `claynel.jp` → `claynel`)
- **納品URL** はカジュアル面談 AI のフロントエンド (例: `https://casual-interview-dev.brainverse-ai.com/<企業ID>`)
- ステータス「スキップ」「未処理」の企業は、CSV の「ホームページURL」列に正しい URL を記入して再実行することで処理を再開できます
- CSV を更新したら、このノートブックを再実行すれば最新状態が反映されます